In [8]:
from sqlalchemy import create_engine, text
import pandas as pd
import matplotlib.pyplot as plt
import os

In [9]:
engine = create_engine(
    f"mysql+mysqlconnector://{os.environ['DB_USER']}:{os.environ['DB_PASSWORD']}@{os.environ['DB_HOST']}/{os.environ['DB_NAME']}"
)

with engine.connect() as conn:
    result = conn.execute(text("SHOW TABLES"))
    tables = [row[0] for row in result]

print("Total Tables:", len(tables))
print("Table Names:")
for table in tables:
    print("-", table)

Total Tables: 2
Table Names:
- location
- trip_details


In [10]:
for table in tables:
       print(f"\n Table: {table}")
       df = pd.read_sql_query(text(f"SELECT COUNT(*) FROM {table}"), engine)
       print(f"{table}", df.iloc[0,0])
       df = pd.read_sql(f"SELECT * FROM {table} limit 5", engine)
       display(df)


 Table: location
location 265


,LocationID,Location,City
0,1,Newark Airport,"Newark, New Jersey"
1,2,Jamaica Bay,Queens
2,3,Allerton/Pelham Gardens,The Bronx
3,4,Alphabet City,Manhattan
4,5,Arden Heights,Staten Island



 Table: trip_details
trip_details 103728


,TripID,PickupTime,DropOffTime,passenger_count,trip_distance,PULocationID,DOLocationID,fare_amount,SurgeFee,Vehicle,Payment_type
0,1,2024-06-01 00:42:50,2024-06-01 01:04:33,1,5.60,79,226,19.5,2.0,UberX,Uber Pay
1,2,2024-06-01 00:06:29,2024-06-01 00:13:22,1,1.72,142,186,8.0,0.0,Uber Black,Cash
2,3,2024-06-01 00:08:05,2024-06-01 00:21:33,1,3.41,229,238,13.0,0.0,Uber Black,Cash
3,4,2024-06-01 00:28:20,2024-06-01 00:37:46,1,1.81,188,35,9.0,0.0,UberX,Cash
4,5,2024-06-01 00:38:05,2024-06-01 00:45:05,1,1.89,100,137,8.0,0.0,Uber Black,Cash


In [11]:
location_df = pd.read_sql_query(text("SELECT * FROM location"), engine)
trip_df = pd.read_sql_query(text("SELECT * FROM trip_details"), engine)

In [12]:
def inspect_data(df, table_name):

    print("\n" + "=" * 60)
    print(f"TABLE: {table_name}")
    print("=" * 60)

    print("Shape:", df.shape)

    print("\nData Types:")
    print(df.dtypes)

    print("\nMissing Values:")
    print(df.isnull().sum())

    print("\nDuplicate Rows:")
    print(df.duplicated().sum())

    print("\nSample Data:")
    display(df.head())


inspect_data(location_df, "location")
inspect_data(trip_df , 'trip_details')


TABLE: location
Shape: (265, 3)

Data Types:
LocationID     int64
Location      object
City          object
dtype: object

Missing Values:
LocationID    0
Location      1
City          2
dtype: int64

Duplicate Rows:
0

Sample Data:


,LocationID,Location,City
0,1,Newark Airport,"Newark, New Jersey"
1,2,Jamaica Bay,Queens
2,3,Allerton/Pelham Gardens,The Bronx
3,4,Alphabet City,Manhattan
4,5,Arden Heights,Staten Island



TABLE: trip_details
Shape: (103728, 11)

Data Types:
TripID                      int64
PickupTime         datetime64[ns]
DropOffTime        datetime64[ns]
passenger_count             int64
trip_distance             float64
PULocationID                int64
DOLocationID                int64
fare_amount               float64
SurgeFee                  float64
Vehicle                    object
Payment_type               object
dtype: object

Missing Values:
TripID             0
PickupTime         0
DropOffTime        0
passenger_count    0
trip_distance      0
PULocationID       0
DOLocationID       0
fare_amount        0
SurgeFee           0
Vehicle            0
Payment_type       0
dtype: int64

Duplicate Rows:
0

Sample Data:


,TripID,PickupTime,DropOffTime,passenger_count,trip_distance,PULocationID,DOLocationID,fare_amount,SurgeFee,Vehicle,Payment_type
0,1,2024-06-01 00:42:50,2024-06-01 01:04:33,1,5.60,79,226,19.5,2.0,UberX,Uber Pay
1,2,2024-06-01 00:06:29,2024-06-01 00:13:22,1,1.72,142,186,8.0,0.0,Uber Black,Cash
2,3,2024-06-01 00:08:05,2024-06-01 00:21:33,1,3.41,229,238,13.0,0.0,Uber Black,Cash
3,4,2024-06-01 00:28:20,2024-06-01 00:37:46,1,1.81,188,35,9.0,0.0,UberX,Cash
4,5,2024-06-01 00:38:05,2024-06-01 00:45:05,1,1.89,100,137,8.0,0.0,Uber Black,Cash


In [13]:
location_null = pd.read_sql_query(text("""select * from location where location is null or city is null """),engine)
display(location_null)

,LocationID,Location,City
0,264,NV,None
1,265,None,None


In [14]:
trip_null = pd.read_sql_query(text("""select * from trip_details where pulocationid in (264 , 265) or dolocationid in (264, 265)"""),engine)
display(trip_null)

,TripID,PickupTime,DropOffTime,passenger_count,trip_distance,PULocationID,DOLocationID,fare_amount,SurgeFee,Vehicle,Payment_type
0,56,2024-06-01 01:49:41,2024-06-01 02:02:04,1,2.50,264,264,11.5,0.00,UberXL,Cash
1,180,2024-06-01 06:54:02,2024-06-01 06:59:41,1,1.10,264,161,6.0,0.00,UberX,Amazon Pay
2,195,2024-06-01 06:00:13,2024-06-01 06:06:33,1,2.00,264,162,8.0,2.26,UberX,Uber Pay
3,239,2024-06-01 06:45:40,2024-06-01 06:51:55,5,1.33,264,264,6.5,0.00,UberX,Cash
4,314,2024-06-01 07:42:43,2024-06-01 08:02:24,1,10.10,264,264,29.5,6.05,UberX,Uber Pay
...,...,...,...,...,...,...,...,...,...,...,...
1105,116708,2024-06-30 22:28:29,2024-06-30 22:58:27,1,15.01,42,265,44.0,9.06,Uber Black,Uber Pay
1106,116800,2024-06-30 22:14:52,2024-06-30 22:22:37,2,1.60,264,264,7.5,0.00,Uber Black,Cash
1107,116809,2024-06-30 22:49:13,2024-06-30 23:14:16,1,17.62,70,265,47.0,0.00,Uber Black,Cash
1108,116913,2024-06-30 23:16:43,2024-06-30 23:28:42,1,2.40,264,264,11.0,0.00,UberX,Cash


* here can't remove location and city which null becuase This isn't a data-quality bug — it's a known placeholder pattern from the standard NYC TLC taxi-zone data this dataset is built on. 264 = "NV" and 265 = blank are the official "Unknown location" and "Outside NYC" zones, not entry errors . 

* so i replace None to Unknown and NV to Outside NYC

In [15]:
with engine.begin() as conn:
    conn.execute(text("""
        UPDATE location SET location = 'Unknown', city = 'Unknown' WHERE locationid = 264;
    """))
    conn.execute(text("""
        UPDATE location SET location = 'Outside NYC', city = 'Unknown' WHERE locationid = 265;
    """))

print("Updated.")

Updated.


In [16]:
pd.read_sql_query(text("""select * from location where city = 'Unknown' """),engine)

,LocationID,Location,City
0,264,Unknown,Unknown
1,265,Outside NYC,Unknown


In [17]:
location_df = pd.read_sql_query(text("""SELECT * FROM location"""), engine)
location_df

,LocationID,Location,City
0,1,Newark Airport,"Newark, New Jersey"
1,2,Jamaica Bay,Queens
2,3,Allerton/Pelham Gardens,The Bronx
3,4,Alphabet City,Manhattan
4,5,Arden Heights,Staten Island
...,...,...,...
260,261,World Trade Center,Manhattan
261,262,Yorkville East,Manhattan
262,263,Yorkville West,Manhattan
263,264,Unknown,Unknown


In [18]:
# Check unique values

def check_unique_values(df, table_name):

    print("\n" + "=" * 70)
    print(f"UNIQUE VALUES — {table_name}")
    print("=" * 70)

    cat_cols = df.select_dtypes(include='object').columns

    for col in cat_cols:
        vals = df[col].unique()

        print(f"\n{col} ({len(vals)} unique):")
        print(sorted(vals))


check_unique_values(location_df, "location")
check_unique_values(trip_df, "trip_details")


UNIQUE VALUES — location

Location (262 unique):
['Allerton/Pelham Gardens', 'Alphabet City', 'Arden Heights', 'Arrochar/Fort Wadsworth', 'Astoria', 'Astoria Park', 'Auburndale', 'Baisley Park', 'Bath Beach', 'Battery Park', 'Battery Park City', 'Bay Ridge', 'Bay Terrace/Fort Totten', 'Bayside', 'Bedford', 'Bedford Park', 'Bellerose', 'Belmont', 'Bensonhurst East', 'Bensonhurst West', 'Bloomfield/Emerson Hill', 'Bloomingdale', 'Boerum Hill', 'Borough Park', 'Breezy Point/Fort Tilden/Riis Beach', 'Briarwood/Jamaica Hills', 'Brighton Beach', 'Broad Channel', 'Bronx Park', 'Bronxdale', 'Brooklyn Heights', 'Brooklyn Navy Yard', 'Brownsville', 'Bushwick North', 'Bushwick South', 'Cambria Heights', 'Canarsie', 'Carroll Gardens', 'Central Harlem', 'Central Harlem North', 'Central Park', 'Charleston/Tottenville', 'Chinatown', 'City Island', 'Claremont/Bathgate', 'Clinton East', 'Clinton Hill', 'Clinton West', 'Co-Op City', 'Cobble Hill', 'College Point', 'Columbia Street', 'Coney Island', 'Co

In [19]:
with engine.begin() as conn:
    conn.execute(text("""
        UPDATE location SET city = 'Bronx' WHERE city = 'The Bronx';
    """))

print("Updated.")

Updated.


In [20]:
# to check update accured or not 
location_df = pd.read_sql_query(text("""SELECT * FROM location"""), engine)
check_unique_values(location_df, "location")


UNIQUE VALUES — location

Location (262 unique):
['Allerton/Pelham Gardens', 'Alphabet City', 'Arden Heights', 'Arrochar/Fort Wadsworth', 'Astoria', 'Astoria Park', 'Auburndale', 'Baisley Park', 'Bath Beach', 'Battery Park', 'Battery Park City', 'Bay Ridge', 'Bay Terrace/Fort Totten', 'Bayside', 'Bedford', 'Bedford Park', 'Bellerose', 'Belmont', 'Bensonhurst East', 'Bensonhurst West', 'Bloomfield/Emerson Hill', 'Bloomingdale', 'Boerum Hill', 'Borough Park', 'Breezy Point/Fort Tilden/Riis Beach', 'Briarwood/Jamaica Hills', 'Brighton Beach', 'Broad Channel', 'Bronx Park', 'Bronxdale', 'Brooklyn Heights', 'Brooklyn Navy Yard', 'Brownsville', 'Bushwick North', 'Bushwick South', 'Cambria Heights', 'Canarsie', 'Carroll Gardens', 'Central Harlem', 'Central Harlem North', 'Central Park', 'Charleston/Tottenville', 'Chinatown', 'City Island', 'Claremont/Bathgate', 'Clinton East', 'Clinton Hill', 'Clinton West', 'Co-Op City', 'Cobble Hill', 'College Point', 'Columbia Street', 'Coney Island', 'Co

In [21]:
# Remove leading/trailing whitespace from object columns

def remove_whitespace(df, table_name):

    cat_cols = df.select_dtypes(include='object').columns

    for col in cat_cols:
        df[col] = df[col].str.strip()

    print(f"{table_name}: Whitespace removed from object columns")


remove_whitespace(location_df, "location")
remove_whitespace(trip_df, "trip_details")

location: Whitespace removed from object columns
trip_details: Whitespace removed from object columns


* check if any negative value exist or not in numeric column

In [22]:
numeric_cols = trip_df.select_dtypes(include=('float64' , 'int64')).columns

negative_report = pd.DataFrame({
    'negative_count': (trip_df[numeric_cols] < 0).sum(),
    'negative_percentage': (
        (trip_df[numeric_cols] < 0).mean() * 100
    ).round(2)
})

# Show only columns having negative values
negative_report = negative_report[
    negative_report['negative_count'] > 0
]

display(negative_report)

,negative_count,negative_percentage


* check wether dropoff < pickup

In [23]:
invalid_time_count = (
    trip_df['DropOffTime'] <= trip_df['PickupTime']
).sum()

print("Rows where dropoff time <= pickup time:", invalid_time_count)

invalid_time_rows = trip_df[
    trip_df['DropOffTime'] <= trip_df['PickupTime']
]

display(invalid_time_rows)

Rows where dropoff time <= pickup time: 2


,TripID,PickupTime,DropOffTime,passenger_count,trip_distance,PULocationID,DOLocationID,fare_amount,SurgeFee,Vehicle,Payment_type
51748,56485,2024-06-17 19:30:55,2024-06-17 19:30:55,2,2.6,75,75,11.5,0.00,UberX,Cash
97930,110167,2024-06-29 17:58:23,2024-06-29 17:58:23,1,2.1,170,170,9.0,2.65,UberXL,Uber Pay


* only 2 record have with same time so i removed both 

In [24]:
with engine.begin() as conn:
    conn.execute(text("""delete from trip_details where tripid in (56485 , 110167);"""))
print('deleted')

deleted


In [25]:
# after delete check and refresh
trip_df = pd.read_sql_query(text("""select * from trip_details """),engine)
trip_df.shape

(103726, 11)

In [26]:
location_df.columns = location_df.columns.str.strip().str.lower().str.replace(' ', '_')
trip_df.columns = trip_df.columns.str.strip().str.lower().str.replace(' ', '_')

* add new columns 

In [27]:
trip_df['total_booking_amount'] = trip_df['fare_amount'] + trip_df['surgefee']
trip_df['pickup_date'] = trip_df['pickuptime'].dt.date
trip_df['pickup_hour'] = trip_df['pickuptime'].dt.hour
trip_df['trip_duration_min'] = (
    trip_df['dropofftime'] - trip_df['pickuptime']
).dt.total_seconds() / 60
trip_df

,tripid,pickuptime,dropofftime,passenger_count,trip_distance,pulocationid,dolocationid,fare_amount,surgefee,vehicle,payment_type,total_booking_amount,pickup_date,pickup_hour,trip_duration_min
0,1,2024-06-01 00:42:50,2024-06-01 01:04:33,1,5.60,79,226,19.5,2.0,UberX,Uber Pay,21.5,2024-06-01,0,21.716667
1,2,2024-06-01 00:06:29,2024-06-01 00:13:22,1,1.72,142,186,8.0,0.0,Uber Black,Cash,8.0,2024-06-01,0,6.883333
2,3,2024-06-01 00:08:05,2024-06-01 00:21:33,1,3.41,229,238,13.0,0.0,Uber Black,Cash,13.0,2024-06-01,0,13.466667
3,4,2024-06-01 00:28:20,2024-06-01 00:37:46,1,1.81,188,35,9.0,0.0,UberX,Cash,9.0,2024-06-01,0,9.433333
4,5,2024-06-01 00:38:05,2024-06-01 00:45:05,1,1.89,100,137,8.0,0.0,Uber Black,Cash,8.0,2024-06-01,0,7.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
103721,116918,2024-06-30 23:17:26,2024-06-30 23:35:00,1,9.40,138,243,27.0,0.0,Uber Black,Uber Pay,27.0,2024-06-30,23,17.566667
103722,116919,2024-06-30 23:08:12,2024-06-30 23:14:02,1,2.00,185,32,7.5,0.0,Uber Green,Cash,7.5,2024-06-30,23,5.833333
103723,116920,2024-06-30 23:38:39,2024-06-30 23:42:56,1,1.30,162,237,6.0,2.9,UberX,Uber Pay,8.9,2024-06-30,23,4.283333
103724,116923,2024-06-30 23:57:00,2024-07-01 00:05:46,1,2.50,142,233,10.0,0.0,Uber Green,Cash,10.0,2024-06-30,23,8.766667


In [28]:
location_df.to_sql(name='clean_location' ,
                     con=engine , 
          if_exists="replace" , 
          index=False,
          chunksize=500)
print('location data saved in mysql')
trip_df.to_sql(name='clean_trip_detail',
                con=engine , 
          if_exists="replace" , 
          index=False,
          chunksize=500)
print('trip detail data save in mysql')

location data saved in mysql
trip detail data save in mysql


In [29]:
original_count_location = pd.read_sql("SELECT COUNT(*) AS cnt FROM clean_location", engine)
print(f"✓ Rows in MySQL  : {original_count_location['cnt'][0]}")
print(f"✓ Rows in df     : {len(location_df)}")
print(f"✓ Match          : {original_count_location['cnt'][0] == len(location_df)}")

original_count_trip = pd.read_sql("SELECT COUNT(*) AS cnt FROM clean_trip_detail", engine)
print(f"✓ Rows in MySQL  : {original_count_trip['cnt'][0]}")
print(f"✓ Rows in df     : {len(trip_df)}")
print(f"✓ Match          : {original_count_trip['cnt'][0] == len(trip_df)}")

✓ Rows in MySQL  : 265
✓ Rows in df     : 265
✓ Match          : True
✓ Rows in MySQL  : 103726
✓ Rows in df     : 103726
✓ Match          : True
